In [1]:
from SelfCal import PipelineWrapper
from astropy.io import fits
import numpy as np
import glob
from SelfCal.SPHERExUtility import interpolate_array, make_chunk_map, make_chunk_mask, visualize_chunk_map, interp_2d_vertical, load_calibration
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 400 # User can set this outside the class if needed
# Import LogNorm
from matplotlib.colors import LogNorm
from tqdm import tqdm
import os
from SelfCal import MakeMap
import gc
from IPython.display import clear_output
import importlib


In [2]:
hdul = fits.open('/data1/SPHEREx/synthetic_maps/det_3_8/synthetic_exposure_1380.fits')

In [3]:
detector = 1
config = {}
config['output_dir'] = '/data1/SPHEREx/synthetic_maps/reprojected_det_3_8'
config['run_name'] = f'nep_det3_6p2arcsec_synthetic_1'
config['resolution_arcsec'] = 6.2

In [6]:
det_BC, det_BW = load_calibration(band=detector, calibration_dir='/home/thomasli/spherex/SPHEREx_Spectral_Calibration')

chunk_map = make_chunk_map(detector, det_BC, num_subchannels=10, num_channels=17,
                   channel_file='/home/thomasli/spherex/spherex_channels.csv')

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:10<00:00, 16.29it/s]


In [7]:
ch = [8]

chunk_valid_mask = make_chunk_mask(ch,  num_subchannels=10, num_channels=17)
det_valid_mask = chunk_valid_mask[chunk_map]
cc = PipelineWrapper.Calibrator(config)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3450/3450 [00:00<00:00, 1580940.54it/s]

Loading reference frame from: /data1/SPHEREx/synthetic_maps/reprojected_det_3_8/nep_det3_6p2arcsec_synthetic_1/ref.fits


In [9]:
cc.setup_lsqr(
    apply_mask=False, 
    apply_weight=False, 
    chunk_map=chunk_map, 
    det_valid_mask=det_valid_mask, 
    max_workers=50, 
    outlier_thresh=20.0,
    ignore_list=[],
    )


Building A, b matrix: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 3450/3450 [15:33<00:00,  3.70it/s]


In [ ]:
cc.apply_lsqr(x0=None, atol=1e-06, btol=1e-06, damp=1e-1, iter_lim=100)

cal_path = cc.save_calibration(cal_file=f'cal_det{detector}_ch{'-'.join(map(str, ch))}.h5')

Solving least squares for 104189016 unknowns with 815642910 equations.
 
LSQR            Least-squares solution of  Ax = b
The matrix A has 815642910 rows and 104189016 columns
damp = 1.00000000000000e-01   calc_var =        0
atol = 1.00e-06                 conlim = 1.00e+08
btol = 1.00e-06               iter_lim =      100
 
   Itn      x[0]       r1norm     r2norm   Compatible    LS      Norm A   Cond A
     0  0.00000e+00   1.112e+04  1.112e+04    1.0e+00  7.9e-03
     1  0.00000e+00   9.116e+03  9.116e+03    8.2e-01  3.4e-02   1.5e+02  1.0e+00
     2  0.00000e+00   9.060e+03  9.060e+03    8.1e-01  1.0e-01   1.6e+02  3.6e+00
     3  0.00000e+00   4.284e+03  4.284e+03    3.9e-01  2.1e-01   2.2e+02  3.9e+01
     4  0.00000e+00   4.087e+03  4.089e+03    3.7e-01  1.4e-02   2.6e+02  4.8e+01
     5  0.00000e+00   4.078e+03  4.080e+03    3.7e-01  3.4e-02   2.7e+02  4.9e+01
     6  0.00000e+00   2.856e+03  2.860e+03    2.6e-01  2.1e-01   3.1e+02  8.7e+01
     7  0.00000e+00   2.518e+03  2.

PermissionError: [Errno 13] Permission denied: '/data1/SPHEREx/synthetic_maps/reprojected_det_3_8/nep_det3_6p2arcsec_synthetic_1/calibration'

In [ ]:
cal_path = cc.save_calibration(cal_file=f'cal_det{detector}_ch{'-'.join(map(str, ch))}.h5')

Calibration saved to /data1/SPHEREx/synthetic_maps/reprojected_det_3_8/nep_det3_6p2arcsec_synthetic_1/calibration/cal_det1_ch8.h5


In [8]:
mm = PipelineWrapper.Mosaicker(config)

mm.load_calibration(cal_path='/data1/SPHEREx/synthetic_maps/reprojected_det_3_8/nep_det3_6p2arcsec_synthetic_1/calibration/cal_det1_ch8.h5')

maps = mm.make_mosaic(
    apply_mask=False, 
    apply_weight=False, 
    chunk_map=chunk_map, 
    det_valid_mask=det_valid_mask, 
    max_workers=30,
    make_std_map=True, 
    apply_sigma_clipping=True, 
    sigma=1.0,
    ignore_list=[21],
    interp_func=interp_2d_vertical
)

mm.save_mosaic(mos_file=f'mosaic_det{detector}_ch{"-".join(map(str, ch))}.fits', overwrite=True)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3450/3450 [00:00<00:00, 861125.26it/s]


Loading reference frame from: /data1/SPHEREx/synthetic_maps/reprojected_det_3_8/nep_det3_6p2arcsec_synthetic_1/ref.fits
Calibration loaded from /data1/SPHEREx/synthetic_maps/reprojected_det_3_8/nep_det3_6p2arcsec_synthetic_1/calibration/cal_det1_ch8.h5


Computing sigma_clip map: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [35:04<00:00, 70.17s/it]


Mosaic saved to /data1/SPHEREx/synthetic_maps/reprojected_det_3_8/nep_det3_6p2arcsec_synthetic_1/mosaic/mosaic_det1_ch8.fits


'/data1/SPHEREx/synthetic_maps/reprojected_det_3_8/nep_det3_6p2arcsec_synthetic_1/mosaic/mosaic_det1_ch8.fits'

In [20]:
O = mm.O
O = O[np.nonzero(O)]

In [30]:
np.std(O)

np.float64(0.028097746375294923)

In [27]:
map = mm.maps['mean_map']
np.std(map[np.nonzero(map)])

np.float32(0.36658046)